In [1]:
import pandas as pd

In [2]:
from google.colab import files

uploaded = files.upload()

Saving SocialMediaUsersDataset.csv to SocialMediaUsersDataset.csv


In [3]:
df = pd.read_csv(list(uploaded.keys())[0])

In [4]:
df.sample(1)

,UserID,Name,Gender,DOB,Interests,City,Country
69993,69994,Gary Hernandez,Female,2001-04-09,"'Health and wellness', 'Gaming', 'Beauty', 'Ou...",Najafgarh,India


In [5]:
import ast

# Convert string interests into list
def clean_interests(x):
    if pd.isna(x):
        return []

    interests = [i.strip().strip("'").strip('"')
                 for i in x.split(",")]

    return interests

df["Interests"] = df["Interests"].apply(clean_interests)

df.sample(3)

,UserID,Name,Gender,DOB,Interests,City,Country
87262,87263,Samuel Graban,Female,1994-05-04,"[Finance and investments, Education and learni...",Montebelluna,Italy
5138,5139,Bonnie Lee,Female,1972-11-08,"[Sports, Fashion, Business and entrepreneurshi...",Rantepao,Indonesia
25777,25778,John Manley,Male,1964-08-13,"[Food and dining, Fashion]",Tejupilco de Hidalgo,Mexico


In [6]:
#Remove Duplicate Interests
df["Interests"] = df["Interests"].apply(
    lambda x: list(set(x))
)

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 80000
Test : 20000


In [8]:
# Create Ground Truth Labels from Test Set

# For every user in test set:

# 1. Randomly remove one interest
# 2. Store removed interest as target label y

In [9]:
import random

random.seed(42)

test_df = test_df.copy()

X_test = []
y_test = []

for interests in test_df["Interests"]:

    if len(interests) < 2:
        continue

    removed = random.choice(interests)

    remaining = interests.copy()
    remaining.remove(removed)

    X_test.append(remaining)
    y_test.append(removed)

In [10]:
from collections import Counter
from itertools import combinations

pair_counter = Counter()

for interests in train_df["Interests"]:

    unique_interests = list(set(interests))

    for pair in combinations(unique_interests, 2):

        pair = tuple(sorted(pair))

        pair_counter[pair] += 1

In [11]:
list(pair_counter.items())[:10]

[(('Photography', 'Politics'), 623),
 (('Food and dining', 'Politics'), 636),
 (('Health and wellness', 'Politics'), 648),
 (('Food and dining', 'Photography'), 627),
 (('Health and wellness', 'Photography'), 678),
 (('Food and dining', 'Health and wellness'), 642),
 (('Cooking', 'Parenting and family'), 687),
 (('Fashion', 'Music'), 1285),
 (('Music', 'Travel'), 692),
 (('Art', 'Music'), 641)]

In [12]:
from collections import defaultdict

graph = defaultdict(Counter)

for (a, b), count in pair_counter.items():

    graph[a][b] += count
    graph[b][a] += count

In [13]:
graph["Nature"]

Counter({'Technology': 665,
         'Food and dining': 640,
         'Pets': 651,
         'Finance and investments': 700,
         'Travel': 654,
         'Science': 634,
         'Social causes and activism': 606,
         'Parenting and family': 694,
         'Books': 721,
         'Cars and automobiles': 639,
         'Education and learning': 628,
         'Movies': 640,
         'Beauty': 617,
         'Cooking': 727,
         'Outdoor activities': 656,
         'Politics': 657,
         'Health and wellness': 645,
         'Gaming': 674,
         'Photography': 613,
         'Fashion': 1297,
         'Art': 676,
         'DIY and crafts': 650,
         'Sports': 670,
         'History': 648,
         'Business and entrepreneurship': 657,
         'Fitness': 692,
         'Music': 673,
         'Gardening': 611})

In [14]:
def recommend(interests, k=10):

    scores = Counter()

    for interest in interests:

        if interest not in graph:
            continue

        scores.update(graph[interest])

    # Remove already known interests
    for interest in interests:
        scores.pop(interest, None)

    recommendations = [
        item
        for item, score
        in scores.most_common(k)
    ]

    return recommendations

In [15]:
recommend(
    ['Nature','Travel'],
    k=5
)

['Fashion', 'Cooking', 'Finance and investments', 'Gaming', 'Books']

In [16]:
def evaluate(X_test, y_test, k=10):

    hits = 0

    for interests, true_interest in zip(
            X_test,
            y_test):

        preds = recommend(interests, k)

        if true_interest in preds:
            hits += 1

    accuracy = hits / len(y_test)

    recall_k = hits / len(y_test)

    precision_k = hits / (len(y_test) * k)

    return {
        "Accuracy": accuracy,
        "Recall@K": recall_k,
        "Precision@K": precision_k
    }

In [17]:
metrics = evaluate(
    X_test,
    y_test,
    k=10
)

print(metrics)

{'Accuracy': 0.4, 'Recall@K': 0.4, 'Precision@K': 0.04}
